# PJM Stack Model from PUDL + CEMS + ICE (DOM Pricing Focus)

This notebook builds a merit-order stack model using **PUDL** generator data, reprices gas units from
`ice_python_cleaned.ice_python_next_day_gas_hourly`, and compares modeled outcomes to:

1. `pjm_cleaned.pjm_lmps_hourly` at `DOMINION HUB` (price fit)
2. `pjm_cleaned.pjm_fuel_mix_hourly` (hourly realized fuel mix check)
3. `power_burns.cems_daily_iso` (daily realized gas-burn check)

The stack itself is PUDL-based. CEMS is used for ex-post validation of what actually ran.


## Notes

- `REGION_FOR_STACK='SOUTH'` is used as the DOM-nearest load region proxy for stack-clearing demand.
- `DOMINION HUB` DA LMP is used as the primary price target for model diagnostics.
- `cems_daily_iso` is ISO-level daily CEMS aggregation, so CEMS validation is performed at **PJM daily** scope.
- This is an analytical prototype, not a production unit-commitment simulator.


In [1]:
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 120)

PARQUET_PATH = "s3://pudl.catalyst.coop/nightly"

TARGET_HUB = "DOMINION HUB"
PRICE_MARKET = "da"
REGION_FOR_STACK = "SOUTH"  # SOUTH is a practical load proxy for Dominion hub pricing.
MODEL_DATE = None  # Set to 'YYYY-MM-DD' to pin a specific operating date.

GAS_PRICE_SERIES = "tetco_m3_cash"  # Alternatives: transco_zone_5_south_cash, hh_cash
CARBON_PRICE_CO2 = 0.0  # USD / ton CO2
MMBTU_PER_BCF = 1_037_000.0


In [2]:
def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "backend" / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root that contains backend/src")


REPO_ROOT = find_repo_root(Path.cwd())
BACKEND_PATH = REPO_ROOT / "backend"
if str(BACKEND_PATH) not in sys.path:
    sys.path.append(str(BACKEND_PATH))

try:
    from src.utils.azure_postgresql import pull_from_db as _pull_from_db
except Exception:
    _pull_from_db = None


def pull_sql(query: str) -> pd.DataFrame:
    if _pull_from_db is not None:
        df = _pull_from_db(query=query)
        if df is None:
            raise RuntimeError("pull_from_db returned None. Verify DB credentials and connectivity.")
        return df

    import psycopg2

    creds = {
        "host": os.getenv("AZURE_POSTGRESQL_DB_HOST"),
        "port": os.getenv("AZURE_POSTGRESQL_DB_PORT"),
        "dbname": os.getenv("AZURE_POSTGRESQL_DB_NAME"),
        "user": os.getenv("AZURE_POSTGRESQL_DB_USER"),
        "password": os.getenv("AZURE_POSTGRESQL_DB_PASSWORD"),
    }
    missing = [k for k, v in creds.items() if not v]
    if missing:
        raise RuntimeError(f"Missing DB env vars: {missing}")

    with psycopg2.connect(**creds) as conn:
        return pd.read_sql(query, conn)


def read_pudl(table_name: str, columns: list[str] | None = None, filters=None) -> pd.DataFrame:
    path = f"{PARQUET_PATH}/{table_name}.parquet"
    return pd.read_parquet(path, columns=columns, filters=filters, dtype_backend="pyarrow")


def normalize_name(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
        .str.lower()
        .str.replace(r"[^a-z0-9\s]", "", regex=True)
        .str.strip()
    )


def add_fuel_bucket(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    fuel = out["fuel_type_code_pudl"].astype(str).str.lower().str.strip()
    pm = out["prime_mover_code"].astype(str).str.upper().str.strip()
    cc_prime_movers = {"CA", "CS", "CT"}

    out["fuel_bucket"] = np.select(
        [
            fuel.eq("nuclear"),
            fuel.eq("coal"),
            fuel.eq("gas") & pm.isin(cc_prime_movers),
            fuel.eq("gas"),
            fuel.eq("oil"),
            fuel.eq("solar"),
            fuel.eq("wind"),
            fuel.eq("hydro"),
            fuel.eq("waste"),
            fuel.eq("storage"),
        ],
        [
            "Nuclear",
            "Coal",
            "Gas CC",
            "Gas CT/ST",
            "Oil",
            "Solar",
            "Wind",
            "Hydro",
            "Biomass",
            "Storage",
        ],
        default="Other",
    )
    return out


INFO:root:CONFIG_DIR: C:\Users\AidanKeaveny\Documents\github\helioscta-pjm-da\backend\src


## 1) Pull Market and Validation Inputs from Warehouse


In [3]:
gas_hourly = pull_sql(
    """
    select
        date::date as date,
        hour_ending::int as hour_ending,
        gas_day::date as gas_day,
        trade_date::date as trade_date,
        hh_cash,
        tetco_m3_cash,
        transco_zone_5_south_cash
    from ice_python_cleaned.ice_python_next_day_gas_hourly
    order by date, hour_ending
    """
)
for c in ["date", "gas_day", "trade_date"]:
    gas_hourly[c] = pd.to_datetime(gas_hourly[c]).dt.date
gas_hourly["hour_ending"] = pd.to_numeric(gas_hourly["hour_ending"], errors="coerce").astype("Int64")

# Use hourly source directly, and keep a daily fallback/aggregation by gas day.
gas_daily = (
    gas_hourly.groupby("gas_day", as_index=False)[
        ["hh_cash", "tetco_m3_cash", "transco_zone_5_south_cash"]
    ]
    .mean()
)

lmps = pull_sql(
    f"""
    select
        datetime,
        date::date as date,
        hour_ending::int as hour_ending,
        hub,
        market,
        lmp_total,
        lmp_system_energy_price,
        lmp_congestion_price,
        lmp_marginal_loss_price
    from pjm_cleaned.pjm_lmps_hourly
    where hub = '{TARGET_HUB}'
      and market in ('da', 'rt', 'dart')
    order by datetime
    """
)
lmps["date"] = pd.to_datetime(lmps["date"]).dt.date

load_da = pull_sql(
    """
    select
        datetime,
        date::date as date,
        hour_ending::int as hour_ending,
        region,
        da_load_mw
    from pjm_cleaned.pjm_load_da_hourly
    order by datetime
    """
)
load_da["date"] = pd.to_datetime(load_da["date"]).dt.date

load_rt = pull_sql(
    """
    select
        datetime,
        date::date as date,
        hour_ending::int as hour_ending,
        region,
        rt_load_mw
    from pjm_cleaned.pjm_load_rt_metered_hourly
    order by datetime
    """
)
load_rt["date"] = pd.to_datetime(load_rt["date"]).dt.date

fuel_mix = pull_sql(
    """
    select
        datetime,
        date::date as date,
        hour_ending::int as hour_ending,
        gas,
        coal,
        nuclear,
        oil,
        hydro,
        wind,
        solar,
        total
    from pjm_cleaned.pjm_fuel_mix_hourly
    order by datetime
    """
)
fuel_mix["date"] = pd.to_datetime(fuel_mix["date"]).dt.date

cems_daily = pull_sql(
    """
    select
        gas_day::date as gas_day,
        iso_region,
        heat_input_mmbtu,
        gas_burn_bcfd
    from power_burns.cems_daily_iso
    where iso_region = 'PJM'
    order by gas_day
    """
)
cems_daily["gas_day"] = pd.to_datetime(cems_daily["gas_day"]).dt.date

lmp_da = lmps[lmps["market"] == PRICE_MARKET].copy()

print(f"ICE hourly rows: {len(gas_hourly):,} | {gas_hourly['date'].min()} -> {gas_hourly['date'].max()}")
print(f"LMP rows ({TARGET_HUB}): {len(lmps):,} | {lmps['date'].min()} -> {lmps['date'].max()}")
print(f"DA load rows: {len(load_da):,} | {load_da['date'].min()} -> {load_da['date'].max()}")
print(f"RT load rows: {len(load_rt):,} | {load_rt['date'].min()} -> {load_rt['date'].max()}")
print(f"Fuel mix rows: {len(fuel_mix):,} | {fuel_mix['date'].min()} -> {fuel_mix['date'].max()}")
print(f"CEMS daily rows: {len(cems_daily):,} | {cems_daily['gas_day'].min()} -> {cems_daily['gas_day'].max()}")


ICE hourly rows: 54,288 | 2020-01-01 -> 2026-03-11
LMP rows (DOMINION HUB): 320,741 | 2014-01-01 -> 2026-03-12
DA load rows: 209,800 | 2020-01-01 -> 2026-03-12
RT load rows: 427,340 | 2014-01-01 -> 2026-03-10
Fuel mix rows: 54,215 | 2020-01-01 -> 2026-03-12
CEMS daily rows: 2,558 | 2019-01-01 -> 2026-01-01


In [4]:
candidate_days = (
    set(load_da.loc[load_da["region"] == REGION_FOR_STACK, "date"])
    & set(lmp_da["date"])
    & set(gas_daily["gas_day"])
)

if not candidate_days:
    raise RuntimeError("No overlapping dates between DA load, DOM hub LMP, and ICE gas series.")

if MODEL_DATE is None:
    model_day = max(candidate_days)
else:
    model_day = pd.to_datetime(MODEL_DATE).date()
    if model_day not in candidate_days:
        min_d = min(candidate_days)
        max_d = max(candidate_days)
        raise ValueError(f"MODEL_DATE={model_day} outside overlap window [{min_d}, {max_d}]")

print(f"Model operating day: {model_day}")
print(f"Stack demand region: {REGION_FOR_STACK}")
print(f"Gas price series: {GAS_PRICE_SERIES}")


Model operating day: 2026-03-12
Stack demand region: SOUTH
Gas price series: tetco_m3_cash


## 2) Build the Stack Fleet from PUDL (No Workbook Inputs)


In [5]:
plants_eia = read_pudl(
    "core_eia860__scd_plants",
    columns=["plant_id_eia", "report_date", "iso_rto_code", "balancing_authority_code_eia"],
)
plants_eia["report_date"] = pd.to_datetime(plants_eia["report_date"])

pjm_reference = plants_eia[
    (plants_eia["iso_rto_code"] == "PJM")
    | (plants_eia["balancing_authority_code_eia"] == "PJM")
].copy()

pjm_reference = (
    pjm_reference.sort_values(["plant_id_eia", "report_date"], ascending=[True, False])
    .drop_duplicates(subset=["plant_id_eia"], keep="first")
    .reset_index(drop=True)
)

pjm_plant_ids = pd.to_numeric(pjm_reference["plant_id_eia"], errors="coerce").dropna().astype(int).unique()

gens_cols = [
    "report_date",
    "plant_id_eia",
    "plant_name_eia",
    "generator_id",
    "unit_id_pudl",
    "utility_id_eia",
    "utility_name_eia",
    "state",
    "balancing_authority_code_eia",
    "operational_status",
    "prime_mover_code",
    "fuel_type_code_pudl",
    "summer_capacity_mw",
    "winter_capacity_mw",
    "unit_heat_rate_mmbtu_per_mwh",
    "generator_retirement_date",
]
gens_all = read_pudl("out_eia__yearly_generators", columns=gens_cols)
gens_all["report_date"] = pd.to_datetime(gens_all["report_date"])

gens_pjm = gens_all[gens_all["plant_id_eia"].isin(pjm_plant_ids)].copy()
latest_year = gens_pjm["report_date"].max()
latest_hr_year = gens_pjm.loc[gens_pjm["unit_heat_rate_mmbtu_per_mwh"].notna(), "report_date"].max()

gens_latest = gens_pjm[
    (gens_pjm["report_date"] == latest_year)
    & (gens_pjm["operational_status"].astype(str).str.lower() == "existing")
].copy()

gens_latest["summer_capacity_mw"] = pd.to_numeric(gens_latest["summer_capacity_mw"], errors="coerce")
gens_latest = gens_latest[gens_latest["summer_capacity_mw"] > 0].copy()

# Pull heat rates from the latest year where heat rates are broadly available.
hr_ref = gens_pjm[gens_pjm["report_date"] == latest_hr_year][
    ["plant_id_eia", "generator_id", "unit_heat_rate_mmbtu_per_mwh"]
].drop_duplicates(["plant_id_eia", "generator_id"], keep="first")

gens_latest = gens_latest.drop(columns=["unit_heat_rate_mmbtu_per_mwh"]).merge(
    hr_ref,
    on=["plant_id_eia", "generator_id"],
    how="left",
)

units = add_fuel_bucket(gens_latest)
units["heat_rate_mmbtu_per_mwh"] = pd.to_numeric(
    units["unit_heat_rate_mmbtu_per_mwh"], errors="coerce"
)

bucket_median_hr = units.groupby("fuel_bucket")["heat_rate_mmbtu_per_mwh"].median()
fallback_hr = {
    "Gas CC": 7.2,
    "Gas CT/ST": 10.5,
    "Coal": 10.2,
    "Oil": 10.5,
    "Nuclear": 10.4,
    "Biomass": 13.0,
    "Other": 9.5,
}

units["heat_rate_mmbtu_per_mwh"] = units["heat_rate_mmbtu_per_mwh"].fillna(
    units["fuel_bucket"].map(bucket_median_hr)
)
units["heat_rate_mmbtu_per_mwh"] = units["heat_rate_mmbtu_per_mwh"].fillna(
    units["fuel_bucket"].map(fallback_hr)
).fillna(0.0)

VOM_DEFAULT = {
    "Nuclear": 2.0,
    "Coal": 4.5,
    "Gas CC": 3.5,
    "Gas CT/ST": 6.0,
    "Oil": 6.5,
    "Hydro": 1.0,
    "Wind": 1.0,
    "Solar": 1.0,
    "Biomass": 4.0,
    "Storage": 2.0,
    "Other": 3.0,
}

CO2_FACTOR = {
    "Gas CC": 0.053,
    "Gas CT/ST": 0.053,
    "Coal": 0.097,
    "Oil": 0.075,
}

units["vom_usd_mwh"] = units["fuel_bucket"].map(VOM_DEFAULT).fillna(3.0)
units["co2_ton_per_mmbtu"] = units["fuel_bucket"].map(CO2_FACTOR).fillna(0.0)

units["fuel_group"] = np.select(
    [
        units["fuel_bucket"].isin(["Gas CC", "Gas CT/ST"]),
        units["fuel_bucket"].eq("Coal"),
        units["fuel_bucket"].eq("Oil"),
        units["fuel_bucket"].eq("Nuclear"),
    ],
    ["gas", "coal", "oil", "nuclear"],
    default="other",
)

must_run_buckets = {"Nuclear", "Hydro", "Wind", "Solar", "Biomass"}
units["is_must_run"] = units["fuel_bucket"].isin(must_run_buckets)

units = units.reset_index(drop=True)
print(f"PJM plant universe: {len(pjm_plant_ids):,} plants")
print(f"Latest generator year: {latest_year.date()} | Heat-rate year: {latest_hr_year.date()}")
print(f"Stack units (existing, >0 MW): {len(units):,}")


PJM plant universe: 2,595 plants
Latest generator year: 2026-01-01 | Heat-rate year: 2024-01-01
Stack units (existing, >0 MW): 3,906


In [6]:
fleet_summary = (
    units.groupby("fuel_bucket", as_index=False)
    .agg(
        units=("generator_id", "count"),
        capacity_mw=("summer_capacity_mw", "sum"),
        hr_median=("heat_rate_mmbtu_per_mwh", "median"),
    )
    .sort_values("capacity_mw", ascending=False)
)

fleet_summary["capacity_share_pct"] = (
    fleet_summary["capacity_mw"] / fleet_summary["capacity_mw"].sum() * 100
).round(2)

display(fleet_summary)


,fuel_bucket,units,capacity_mw,hr_median,capacity_share_pct
2,Gas CC,305,60354.101562,7.228644,28.5
1,Coal,100,36427.300781,10.979551,17.2
3,Gas CT/ST,673,36109.101562,11.361483,17.05
5,Nuclear,32,33489.601562,10.4,15.81
8,Solar,1247,17824.5,0.0,8.42
9,Wind,143,11917.200195,0.0,5.63
4,Hydro,313,8443.5,0.0,3.99
6,Oil,499,5024.299805,11.320438,2.37
0,Biomass,555,1784.099976,16.560347,0.84
7,Other,39,428.799988,9.5,0.2


## 3) Dispatch Engine


In [7]:
NON_GAS_FUEL_PRICE = {
    "coal": 2.50,
    "oil": 14.00,
    "nuclear": 0.70,
    "other": 0.00,
}


def build_hourly_stack(units_df: pd.DataFrame, gas_price: float, carbon_price: float = 0.0) -> pd.DataFrame:
    stack = units_df.copy()

    stack["fuel_price_mmbtu"] = stack["fuel_group"].map(NON_GAS_FUEL_PRICE).fillna(0.0)
    stack.loc[stack["fuel_group"] == "gas", "fuel_price_mmbtu"] = float(gas_price)

    stack["fuel_cost_usd_mwh"] = stack["heat_rate_mmbtu_per_mwh"] * stack["fuel_price_mmbtu"]
    stack["co2_cost_usd_mwh"] = (
        stack["heat_rate_mmbtu_per_mwh"] * stack["co2_ton_per_mmbtu"] * float(carbon_price)
    )
    stack["mc_usd_mwh"] = stack["fuel_cost_usd_mwh"] + stack["vom_usd_mwh"] + stack["co2_cost_usd_mwh"]

    must_run = stack[stack["is_must_run"]].sort_values("mc_usd_mwh")
    dispatchable = stack[~stack["is_must_run"]].sort_values("mc_usd_mwh")
    ordered = pd.concat([must_run, dispatchable], ignore_index=True)

    ordered["cum_mw"] = ordered["summer_capacity_mw"].cumsum()
    ordered["cum_prev_mw"] = ordered["cum_mw"] - ordered["summer_capacity_mw"]
    return ordered


def clear_stack(stack_df: pd.DataFrame, demand_mw: float) -> dict:
    demand = max(float(demand_mw), 0.0)
    disp = stack_df.copy()

    disp["dispatched_mw"] = np.clip(
        demand - disp["cum_prev_mw"],
        0,
        disp["summer_capacity_mw"],
    )

    if demand <= 0:
        return {
            "price": 0.0,
            "shortage_mw": 0.0,
            "marginal_fuel": "N/A",
            "marginal_plant": "N/A",
            "dispatch_by_fuel": disp.groupby("fuel_bucket")["dispatched_mw"].sum().to_dict(),
            "modeled_gas_heat_input_mmbtu": 0.0,
        }

    mask = disp["cum_mw"] >= demand
    if mask.any():
        idx = mask.idxmax()
        marginal = disp.loc[idx]
        price = float(marginal["mc_usd_mwh"])
        shortage = 0.0
    else:
        marginal = disp.iloc[-1]
        price = np.nan
        shortage = float(demand - disp["cum_mw"].iloc[-1])

    gas_heat_input = float(
        (
            disp.loc[disp["fuel_group"] == "gas", "dispatched_mw"]
            * disp.loc[disp["fuel_group"] == "gas", "heat_rate_mmbtu_per_mwh"]
        ).sum()
    )

    return {
        "price": price,
        "shortage_mw": shortage,
        "marginal_fuel": str(marginal["fuel_bucket"]),
        "marginal_plant": str(marginal["plant_name_eia"]),
        "dispatch_by_fuel": disp.groupby("fuel_bucket")["dispatched_mw"].sum().to_dict(),
        "modeled_gas_heat_input_mmbtu": gas_heat_input,
    }


def get_hourly_gas_for_day(model_day, gas_series: str) -> pd.DataFrame:
    day_rows = gas_hourly.loc[gas_hourly["gas_day"] == model_day, ["hour_ending", gas_series]].dropna().copy()
    day_rows["hour_ending"] = pd.to_numeric(day_rows["hour_ending"], errors="coerce").astype("Int64")
    day_rows = day_rows.dropna(subset=["hour_ending"])

    if day_rows.empty:
        fallback = gas_daily.loc[gas_daily["gas_day"] <= model_day].sort_values("gas_day").tail(1)
        if fallback.empty:
            raise RuntimeError(f"No gas data available on or before {model_day}")
        p = float(fallback.iloc[0][gas_series])
        return pd.DataFrame({"hour_ending": list(range(1, 25)), "gas_price": [p] * 24})

    day_rows = day_rows.drop_duplicates(subset=["hour_ending"], keep="last")
    day_rows = day_rows.rename(columns={gas_series: "gas_price"})

    if day_rows["hour_ending"].nunique() < 24:
        all_hours = pd.DataFrame({"hour_ending": list(range(1, 25))})
        fill_price = float(day_rows["gas_price"].mean())
        day_rows = all_hours.merge(day_rows, on="hour_ending", how="left")
        day_rows["gas_price"] = day_rows["gas_price"].fillna(fill_price)

    return day_rows.sort_values("hour_ending").reset_index(drop=True)


def run_stack_for_day(
    model_day,
    region: str,
    market: str = "da",
    gas_series: str = "tetco_m3_cash",
    gas_adder: float = 0.0,
) -> pd.DataFrame:
    if market not in {"da", "rt"}:
        raise ValueError("market must be 'da' or 'rt'")

    load_df = load_da if market == "da" else load_rt
    load_col = "da_load_mw" if market == "da" else "rt_load_mw"

    day_load = load_df.loc[
        (load_df["date"] == model_day) & (load_df["region"] == region),
        ["hour_ending", load_col],
    ].copy()
    if day_load.empty:
        return pd.DataFrame()

    day_load["hour_ending"] = pd.to_numeric(day_load["hour_ending"], errors="coerce").astype("Int64")
    day_load = day_load.dropna(subset=["hour_ending"]).sort_values("hour_ending")

    gas_profile = get_hourly_gas_for_day(model_day, gas_series)
    joined = day_load.merge(gas_profile, on="hour_ending", how="left")

    if joined["gas_price"].isna().all():
        fallback = gas_daily.loc[gas_daily["gas_day"] <= model_day].sort_values("gas_day").tail(1)
        if fallback.empty:
            raise RuntimeError(f"No gas daily fallback available for {model_day}")
        joined["gas_price"] = float(fallback.iloc[0][gas_series])

    joined["gas_price"] = joined["gas_price"].fillna(float(joined["gas_price"].mean())) + float(gas_adder)

    fuel_cols = ["Nuclear", "Coal", "Gas CC", "Gas CT/ST", "Oil", "Hydro", "Wind", "Solar", "Biomass", "Storage", "Other"]
    records = []

    for row in joined.itertuples(index=False):
        he = int(row.hour_ending)
        demand = float(getattr(row, load_col))
        gas_p = float(row.gas_price)

        stack = build_hourly_stack(units, gas_price=gas_p, carbon_price=CARBON_PRICE_CO2)
        cleared = clear_stack(stack, demand_mw=demand)

        rec = {
            "date": model_day,
            "hour_ending": he,
            "region": region,
            "market": market,
            "demand_mw": demand,
            "gas_price_usd_mmbtu": gas_p,
            "modeled_price_usd_mwh": cleared["price"],
            "modeled_shortage_mw": cleared["shortage_mw"],
            "marginal_fuel": cleared["marginal_fuel"],
            "marginal_plant": cleared["marginal_plant"],
            "modeled_gas_heat_input_mmbtu": cleared["modeled_gas_heat_input_mmbtu"],
        }
        for fuel in fuel_cols:
            rec[f"dispatch_{fuel.lower().replace(' ', '_').replace('/', '_')}_mw"] = float(
                cleared["dispatch_by_fuel"].get(fuel, 0.0)
            )
        records.append(rec)

    return pd.DataFrame(records).sort_values("hour_ending").reset_index(drop=True)


## 4) DOM Pricing Fit (Ex-Ante)


In [8]:
dom_model = run_stack_for_day(
    model_day=model_day,
    region=REGION_FOR_STACK,
    market=PRICE_MARKET,
    gas_series=GAS_PRICE_SERIES,
)

if dom_model.empty:
    raise RuntimeError(f"No model output for day={model_day}, region={REGION_FOR_STACK}")

dom_lmp_day = lmp_da.loc[lmp_da["date"] == model_day, ["hour_ending", "lmp_total"]].copy()
dom_lmp_day = dom_lmp_day.rename(columns={"lmp_total": "dom_da_lmp_usd_mwh"})

dom_compare = dom_model.merge(dom_lmp_day, on="hour_ending", how="left")
dom_compare["price_error_usd_mwh"] = dom_compare["modeled_price_usd_mwh"] - dom_compare["dom_da_lmp_usd_mwh"]
dom_compare["abs_error_usd_mwh"] = dom_compare["price_error_usd_mwh"].abs()

mae = dom_compare["abs_error_usd_mwh"].mean()
bias = dom_compare["price_error_usd_mwh"].mean()
cor = dom_compare[["modeled_price_usd_mwh", "dom_da_lmp_usd_mwh"]].corr().iloc[0, 1]

print(f"DOM DA comparison for {model_day} (region proxy={REGION_FOR_STACK})")
print(f"MAE:  {mae:,.2f} USD/MWh")
print(f"Bias: {bias:,.2f} USD/MWh")
print(f"Corr: {cor:,.3f}")

display(
    dom_compare[
        [
            "hour_ending",
            "demand_mw",
            "gas_price_usd_mmbtu",
            "modeled_price_usd_mwh",
            "dom_da_lmp_usd_mwh",
            "price_error_usd_mwh",
            "marginal_fuel",
        ]
    ]
)


DOM DA comparison for 2026-03-12 (region proxy=SOUTH)
MAE:  46.58 USD/MWh
Bias: -46.58 USD/MWh
Corr: nan


,hour_ending,demand_mw,gas_price_usd_mmbtu,modeled_price_usd_mwh,dom_da_lmp_usd_mwh,price_error_usd_mwh,marginal_fuel
0,1,12400.7,2.578267,1.0,33.840154,-32.840154,Wind
1,2,11997.1,2.578267,1.0,33.432441,-32.432441,Wind
2,3,11609.7,2.578267,1.0,30.584397,-29.584397,Wind
3,4,11427.6,2.578267,1.0,32.451066,-31.451066,Wind
4,5,11374.5,2.578267,1.0,37.807882,-36.807882,Wind
5,6,11837.6,2.578267,1.0,45.504968,-44.504968,Wind
6,7,12691.6,2.578267,1.0,60.782773,-59.782773,Wind
7,8,13280.1,2.578267,1.0,77.642472,-76.642472,Wind
8,9,13223.0,2.578267,1.0,64.836557,-63.836557,Wind
9,10,13312.1,2.578267,1.0,44.268829,-43.268829,Wind


In [9]:
plot_df = dom_compare.copy()
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=plot_df["hour_ending"],
        y=plot_df["modeled_price_usd_mwh"],
        mode="lines+markers",
        name="Modeled Stack Price",
    )
)
fig.add_trace(
    go.Scatter(
        x=plot_df["hour_ending"],
        y=plot_df["dom_da_lmp_usd_mwh"],
        mode="lines+markers",
        name="DOMINION HUB DA LMP",
    )
)
fig.update_layout(
    title=f"DOM Pricing Fit ({model_day})",
    xaxis_title="Hour Ending",
    yaxis_title="USD/MWh",
    xaxis=dict(dtick=1),
    template="plotly_white",
    height=450,
)
fig.show()


In [10]:
scenario_rows = []
for gas_shift in [-1.0, -0.5, 0.0, 0.5, 1.0]:
    scen = run_stack_for_day(
        model_day=model_day,
        region=REGION_FOR_STACK,
        market=PRICE_MARKET,
        gas_series=GAS_PRICE_SERIES,
        gas_adder=gas_shift,
    )
    scenario_rows.append(
        {
            "gas_shift_usd_mmbtu": gas_shift,
            "avg_modeled_price_usd_mwh": scen["modeled_price_usd_mwh"].mean(),
            "peak_modeled_price_usd_mwh": scen["modeled_price_usd_mwh"].max(),
            "avg_marginal_gas_share_pct": (
                scen["marginal_fuel"].isin(["Gas CC", "Gas CT/ST"]).mean() * 100
            ),
        }
    )

gas_sensitivity = pd.DataFrame(scenario_rows)
display(gas_sensitivity)

fig = px.line(
    gas_sensitivity,
    x="gas_shift_usd_mmbtu",
    y="avg_modeled_price_usd_mwh",
    markers=True,
    title="DOM Stack Price Sensitivity to Gas Price Shift",
)
fig.update_layout(template="plotly_white", height=420)
fig.show()


,gas_shift_usd_mmbtu,avg_modeled_price_usd_mwh,peak_modeled_price_usd_mwh,avg_marginal_gas_share_pct
0,-1.0,1.0,1.0,0.0
1,-0.5,1.0,1.0,0.0
2,0.0,1.0,1.0,0.0
3,0.5,1.0,1.0,0.0
4,1.0,1.0,1.0,0.0


## 5) Ex-Post Checks: Fuel Mix and CEMS

First, run an RTO-level stack day and compare modeled fuel dispatch to actual PJM hourly fuel mix.
Then compare modeled daily gas burn to CEMS daily gas burn (`power_burns.cems_daily_iso`).


In [11]:
rto_model = run_stack_for_day(
    model_day=model_day,
    region="RTO",
    market="da",
    gas_series=GAS_PRICE_SERIES,
)

fuel_mix_day = fuel_mix.loc[
    fuel_mix["date"] == model_day,
    ["hour_ending", "gas", "coal", "nuclear", "oil", "wind", "solar", "hydro", "total"],
].copy()
for c in ["gas", "coal", "nuclear", "oil", "wind", "solar", "hydro", "total"]:
    fuel_mix_day[c] = pd.to_numeric(fuel_mix_day[c], errors="coerce")

fuel_compare = rto_model.merge(fuel_mix_day, on="hour_ending", how="left")
fuel_compare["modeled_gas_mw"] = (
    fuel_compare["dispatch_gas_cc_mw"] + fuel_compare["dispatch_gas_ct_st_mw"]
)
fuel_compare["modeled_coal_mw"] = fuel_compare["dispatch_coal_mw"]
fuel_compare["modeled_gas_share"] = fuel_compare["modeled_gas_mw"] / fuel_compare["demand_mw"]
fuel_compare["actual_gas_share"] = fuel_compare["gas"] / fuel_compare["total"]

display(
    fuel_compare[
        [
            "hour_ending",
            "demand_mw",
            "modeled_gas_mw",
            "gas",
            "modeled_coal_mw",
            "coal",
            "modeled_gas_share",
            "actual_gas_share",
        ]
    ]
)


,hour_ending,demand_mw,modeled_gas_mw,gas,modeled_coal_mw,coal,modeled_gas_share,actual_gas_share
0,1,76245.7,2357.623409,33914.0,0.0,13007.0,0.030921,0.389060
1,2,74516.5,628.420303,33463.0,0.0,12258.0,0.008433,0.390540
2,3,73425.7,0.000000,33385.0,0.0,10944.0,0.000000,0.395895
3,4,73856.9,0.000000,33579.0,0.0,11272.0,0.000000,0.398587
4,5,74592.6,704.520303,33621.0,0.0,11875.0,0.009445,0.398306
5,6,78463.7,4575.629634,34302.0,0.0,12984.0,0.058315,0.400546
6,7,86680.2,12792.123433,36910.0,0.0,13392.0,0.147578,0.415452
7,8,93278.2,19390.126545,39176.0,0.0,13772.0,0.207874,0.418252
8,9,94070.2,20182.128102,40180.0,0.0,14054.0,0.214543,0.416213
9,10,94492.7,20604.631230,38619.0,0.0,14057.0,0.218055,0.398311


In [12]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=fuel_compare["hour_ending"], y=fuel_compare["modeled_gas_mw"], mode="lines+markers", name="Modeled Gas MW"))
fig.add_trace(go.Scatter(x=fuel_compare["hour_ending"], y=fuel_compare["gas"], mode="lines+markers", name="Actual Gas MW (PJM fuel mix)"))
fig.add_trace(go.Scatter(x=fuel_compare["hour_ending"], y=fuel_compare["modeled_coal_mw"], mode="lines+markers", name="Modeled Coal MW"))
fig.add_trace(go.Scatter(x=fuel_compare["hour_ending"], y=fuel_compare["coal"], mode="lines+markers", name="Actual Coal MW (PJM fuel mix)"))
fig.update_layout(
    title=f"RTO Modeled vs Actual Fuel Mix ({model_day})",
    xaxis_title="Hour Ending",
    yaxis_title="MW",
    xaxis=dict(dtick=1),
    template="plotly_white",
    height=500,
)
fig.show()


In [13]:
def run_daily_stack_series(days: list, region: str = "RTO", market: str = "da") -> pd.DataFrame:
    out = []
    for d in days:
        day_res = run_stack_for_day(
            model_day=d,
            region=region,
            market=market,
            gas_series=GAS_PRICE_SERIES,
        )
        if day_res.empty:
            continue
        out.append(
            {
                "date": d,
                "modeled_gas_bcf": day_res["modeled_gas_heat_input_mmbtu"].sum() / MMBTU_PER_BCF,
                "modeled_avg_price_usd_mwh": day_res["modeled_price_usd_mwh"].mean(),
                "modeled_peak_price_usd_mwh": day_res["modeled_price_usd_mwh"].max(),
                "modeled_daily_load_mwh": day_res["demand_mw"].sum(),
            }
        )
    return pd.DataFrame(out)


cems_pjm = cems_daily[cems_daily["iso_region"] == "PJM"].copy()

overlap_days = sorted(
    set(cems_pjm["gas_day"])
    & set(load_da.loc[load_da["region"] == "RTO", "date"])
    & set(gas_daily["gas_day"])
)

if len(overlap_days) == 0:
    raise RuntimeError("No overlap between CEMS daily, RTO DA load, and ICE gas series.")

backtest_days = overlap_days[-45:]
rto_daily = run_daily_stack_series(backtest_days, region="RTO", market="da")

cems_compare = rto_daily.merge(
    cems_pjm[["gas_day", "gas_burn_bcfd", "heat_input_mmbtu"]].rename(columns={"gas_day": "date"}),
    on="date",
    how="left",
)

cems_compare["gas_error_bcf"] = cems_compare["modeled_gas_bcf"] - cems_compare["gas_burn_bcfd"]
cems_compare["gas_abs_error_bcf"] = cems_compare["gas_error_bcf"].abs()

print(f"Backtest window: {min(backtest_days)} -> {max(backtest_days)} ({len(backtest_days)} days requested)")
print(f"Rows modeled: {len(cems_compare):,}")
print(f"Gas-burn MAE: {cems_compare['gas_abs_error_bcf'].mean():.3f} Bcf/day")
print(f"Gas-burn Bias: {cems_compare['gas_error_bcf'].mean():.3f} Bcf/day")

display(cems_compare.tail(20))


Backtest window: 2025-11-18 -> 2026-01-01 (45 days requested)
Rows modeled: 45
Gas-burn MAE: 10.841 Bcf/day
Gas-burn Bias: -10.841 Bcf/day


,date,modeled_gas_bcf,modeled_avg_price_usd_mwh,modeled_peak_price_usd_mwh,modeled_daily_load_mwh,gas_burn_bcfd,heat_input_mmbtu,gas_error_bcf,gas_abs_error_bcf
25,2025-12-13,0.000000,33.220631,37.782218,2434224.3,13.800987,1.418742e+07,-13.800987,13.800987
26,2025-12-14,0.578772,54.362331,83.011719,2640120.3,14.637819,1.504768e+07,-14.059047,14.059047
27,2025-12-15,1.292961,69.135175,82.622979,2856326.5,17.668291,1.816300e+07,-16.375331,16.375331
28,2025-12-16,1.645861,36.841183,52.657407,2614015.1,15.878972,1.632358e+07,-14.233111,14.233111
29,2025-12-17,3.423217,28.865272,29.951271,2406805.1,12.905107,1.326645e+07,-9.481891,9.481891
30,2025-12-18,3.081773,28.043040,29.911909,2299180.2,11.562551,1.188630e+07,-8.480777,8.480777
31,2025-12-19,2.865611,28.256227,29.967274,2307174.6,11.081746,1.139203e+07,-8.216135,8.216135
32,2025-12-20,3.120260,26.814388,27.504359,2274851.5,11.685400,1.201259e+07,-8.565140,8.565140
33,2025-12-21,2.791969,26.592872,27.581399,2221726.2,11.384473,1.170324e+07,-8.592504,8.592504
34,2025-12-22,4.061720,26.620236,27.943106,2418288.8,12.683480,1.303862e+07,-8.621760,8.621760


In [14]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=cems_compare["date"],
        y=cems_compare["modeled_gas_bcf"],
        mode="lines+markers",
        name="Modeled Gas Burn (RTO stack)",
    )
)
fig.add_trace(
    go.Scatter(
        x=cems_compare["date"],
        y=cems_compare["gas_burn_bcfd"],
        mode="lines+markers",
        name="CEMS Gas Burn (PJM)",
    )
)
fig.update_layout(
    title="Modeled vs CEMS Daily Gas Burn",
    xaxis_title="Date",
    yaxis_title="Bcf/day",
    template="plotly_white",
    height=460,
)
fig.show()


In [15]:
out_dir = Path("outputs/pjm-pudl-cems-stack")
out_dir.mkdir(parents=True, exist_ok=True)

dom_compare.to_csv(out_dir / "dom_price_fit_hourly.csv", index=False)
fuel_compare.to_csv(out_dir / "rto_fuel_mix_compare_hourly.csv", index=False)
cems_compare.to_csv(out_dir / "rto_cems_daily_compare.csv", index=False)

print(f"Wrote outputs to: {out_dir.resolve()}")


Wrote outputs to: C:\Users\AidanKeaveny\Documents\github\helioscta-pjm-da\notebooks\pjm-pudl-cems-stack\outputs\pjm-pudl-cems-stack


## 6) Suggested Next Extensions

1. Replace static non-gas fuel prices with PUDL EIA923 fuel-cost curves by fuel and season.
2. Add outage/availability constraints from PJM outage feeds before stack clearing.
3. Calibrate must-run and minimum-load assumptions to improve DOM DA fit.
4. Add DA vs RT spread diagnostics (`market='dart'`) with gas and load forecast errors.
5. If a unit-level CEMS table is added, upgrade validation from ISO-daily to unit-hourly dispatch matching.
